<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/Step_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 10. HITL и Active Learning — подробная методичка

Эта методичка продолжает Этапы 1–9. Предполагается, что у вас:
- `docker compose ps` — `db`, `redis`, `qdrant`, `ollama` в `healthy`, `app` в `healthy`;
- `/health` — 200, все `services.*.healthy: true`;
- `perform_translation` работает, `/translate`, `/translate/multi`, `/translate/batch`, `/translate/ensemble` — 200;
- `critic_score` в ответе `/translate` для режимов с Critic (`agents_only`, `tm_hybrid_critic`, `tm_hybrid_editor`, `full`);
- **в `perform_translation` уже есть вызов `ActiveLearningManager.add_uncertain_example(...)`** (вставлен в Этапе 9.3) — но `ActiveLearningManager` пока **не создан**, значит, если вы запустите `/translate` сейчас, он упадёт с `ImportError`.

Если чего-то нет — вернитесь к Этапу 9.

**Цель этапа:** создать `ActiveLearningManager` и `FeedbackHandler`, которые обрабатывают очередь неуверенных переводов, подтверждение/отклонение, сохранение корректировок в TM — атомарно с обновлением статуса.

**Недель:** 1.

**Что вы получите в конце этапа:**

- `backend/hitl/active_learning.py` — `ActiveLearningManager` с 6 методами (`add_uncertain_example`, `get_pending_examples`, `count_pending`, `confirm_correction`, `reject_example`, `get_stats`) + хелпер `_to_dict`.
- `backend/hitl/feedback.py` — `FeedbackHandler.save_correction` (прямой путь без статуса).
- `backend/api/routes/hitl.py` — 5 эндпоинтов (`/hitl/save-correction`, `/hitl/uncertain`, `/hitl/uncertain/{id}/confirm`, `/hitl/uncertain/{id}/reject`, `/hitl/uncertain/stats`), 2 Pydantic-модели (`CorrectionRequest`, `ConfirmCorrectionRequest`).
- `backend/api/main.py` — регистрация `hitl.router`.
- Тесты `tests/unit/test_active_learning.py`.

**Критерии готовности:**

- [ ] После `/translate` с `critic_score < 7` (или `validation_passed=False`) появляется запись в `uncertain_examples` со `status="pending"`.
- [ ] `add_uncertain_example` **не создаёт дубль**: повторный вызов для того же `(source_text, source_lang, target_lang, status="pending")` возвращает `existing.id`.
- [ ] `confirm_correction` атомарен: `INSERT в TM` и `UPDATE status="confirmed"` — в одной транзакции.
- [ ] При падении в середине транзакции TM-запись НЕ создана (проверяется тестом с моком).
- [ ] `confirm_correction` возвращает `tm_inserted=False` для существующей TM-пары, но пример всё равно `confirmed`.
- [ ] `reject_example` ставит `status="rejected"`, не пишет в TM.
- [ ] `get_stats()` — `{pending, confirmed, rejected}` одним запросом.
- [ ] Все 5 роутов под `require_editor`.
- [ ] Ветка `week-10` создана, коммит сделан.

> **Важно.** В Этапе 9.3 в `perform_translation` уже стоит:
> ```python
> if (settings.ACTIVE_LEARNING_ENABLED
>         and critic_score is not None
>         and (critic_score < CRITIC_PASS_THRESHOLD or not validation_passed)):
>     await ActiveLearningManager.add_uncertain_example(...)
> ```
> Импорт `from backend.hitl.active_learning import ActiveLearningManager` уже есть в `pipeline.py`. Сейчас создаём этот модуль — и связка оживает. Тесты Этапа 9 в части `/translate` до этого момента либо падали (если запускали), либо были с `ACTIVE_LEARNING_ENABLED=false` в `.env` (в `.env` из Этапа 1 стоит `false` — значит, ветка не срабатывала, `add_uncertain_example` не вызывался).

---

## 10.0. Подготовка: ветка

```powershell
cd D:\RUNG
git checkout -b week-10
```

Проверьте, что `backend/hitl/__init__.py` есть (создан в Этапе 2.1 как пустой файл):

```powershell
dir backend\hitl\
```

**Что вы должны увидеть** (`__init__.py` — 0 байт, `active_learning.py` и `feedback.py` пока отсутствуют или пустые):

```
    Directory: D:\RUNG\backend\hitl

Mode    LastWriteTime    Length Name
----    -------------    ------ ----
-a---   ...                0 __init__.py
```

Если `__init__.py` нет — создайте (`type nul > backend\hitl\__init__.py`).

**Проверьте `.env`:** если там `ACTIVE_LEARNING_ENABLED=false`, на этом этапе **переключите на `true`** — иначе `perform_translation` не будет вызывать `add_uncertain_example`, и вы не увидите связку в действии. После переключения `docker compose restart app`.

---

## 10.1. `backend/hitl/active_learning.py` — `ActiveLearningManager`

**Что это:** менеджер очереди неуверенных переводов. Методы:
- `add_uncertain_example` — добавить в очередь (с проверкой на дубль).
- `get_pending_examples` — список `pending` с фильтрами.
- `count_pending` — количество `pending`.
- `confirm_correction` — атомарно: вставить в TM + поставить `status="confirmed"`.
- `reject_example` — поставить `status="rejected"`.
- `get_stats` — `{pending, confirmed, rejected}`.

**Зачем это нужно:** Critic ставит `critic_score < 7` — перевод сомнительный. Автоматический pipeline не может сам решить, как его улучшить. Нужен человек (редактор), который посмотрит и либо подтвердит исправление, либо отклонит. Очередь `uncertain_examples` — механизм «человек в цикле» (Human-in-the-Loop). Подтверждённые исправления попадают в TM — следующая генерация использует их как контекст.

Создайте `backend/hitl/active_learning.py`:

```python
# backend/hitl/active_learning.py
import logging
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from sqlalchemy import case, func, select

from backend.core.exceptions import UncertainExampleNotFoundError
from backend.core.models import TMEntry
from backend.data.translation_memory import tm_manager
from backend.db.models import UncertainExample
from backend.db.session import AsyncSessionLocal

logger = logging.getLogger(__name__)


def _to_dict(e: UncertainExample) -> Dict[str, Any]:
    return {
        "id": e.id,
        "source_text": e.source_text,
        "source_lang": e.source_lang,
        "target_lang": e.target_lang,
        "machine_translation": e.machine_translation,
        "critic_score": e.critic_score,
        "validation_passed": e.validation_passed,
        "created_at": e.created_at.isoformat(),
    }


class ActiveLearningManager:
    """Manage uncertain translation examples for human review."""

    @staticmethod
    async def add_uncertain_example(
        source_text: str,
        source_lang: str,
        target_lang: str,
        machine_translation: str,
        critic_score: Optional[float] = None,
        validation_passed: bool = False,
    ) -> str:
        async with AsyncSessionLocal() as session:
            try:
                stmt = select(UncertainExample).where(
                    UncertainExample.source_text == source_text,
                    UncertainExample.source_lang == source_lang,
                    UncertainExample.target_lang == target_lang,
                    UncertainExample.status == "pending",
                ).limit(1)
                existing = (await session.execute(stmt)).scalar_one_or_none()
                if existing is not None:
                    logger.debug(
                        "Pending example already exists: %s...", source_text[:50]
                    )
                    return existing.id

                example = UncertainExample(
                    id=str(uuid.uuid4()),
                    source_text=source_text,
                    source_lang=source_lang,
                    target_lang=target_lang,
                    machine_translation=machine_translation,
                    critic_score=critic_score,
                    validation_passed=validation_passed,
                    status="pending",
                )
                session.add(example)
                await session.commit()
                logger.info("Uncertain example added: %s", example.id)
                return example.id
            except Exception as e:
                logger.error("Failed to add uncertain example: %s", e, exc_info=True)
                await session.rollback()
                raise

    @staticmethod
    async def get_pending_examples(
        source_lang: Optional[str] = None,
        target_lang: Optional[str] = None,
        limit: int = 10,
        offset: int = 0,
    ) -> List[Dict[str, Any]]:
        async with AsyncSessionLocal() as session:
            try:
                stmt = select(UncertainExample).where(
                    UncertainExample.status == "pending"
                )
                if source_lang:
                    stmt = stmt.where(UncertainExample.source_lang == source_lang)
                if target_lang:
                    stmt = stmt.where(UncertainExample.target_lang == target_lang)
                stmt = (
                    stmt.order_by(UncertainExample.created_at.desc())
                    .offset(offset)
                    .limit(limit)
                )
                examples = (await session.execute(stmt)).scalars().all()
                return [_to_dict(e) for e in examples]
            except Exception as e:
                logger.error(
                    "Failed to fetch pending examples: %s", e, exc_info=True
                )
                return []

    @staticmethod
    async def count_pending(
        source_lang: Optional[str] = None,
        target_lang: Optional[str] = None,
    ) -> int:
        async with AsyncSessionLocal() as session:
            try:
                stmt = select(func.count()).select_from(UncertainExample).where(
                    UncertainExample.status == "pending"
                )
                if source_lang:
                    stmt = stmt.where(UncertainExample.source_lang == source_lang)
                if target_lang:
                    stmt = stmt.where(UncertainExample.target_lang == target_lang)
                return (await session.execute(stmt)).scalar_one()
            except Exception as e:
                logger.error(
                    "Failed to count pending examples: %s", e, exc_info=True
                )
                return 0

    @staticmethod
    async def confirm_correction(
        example_id: str,
        corrected_translation: str,
    ) -> bool:
        """
        Confirm a pending example and persist the correction into TM.

        Both writes happen in a single transaction:
          - INSERT into ``translation_memories`` via
            ``tm_manager.insert_in_session`` (which does NOT commit), and
          - UPDATE of the ``uncertain_examples`` row to ``"confirmed"``.

        The session commits once. Either the TM row and the status change
        are both persisted, or neither is.
        """
        inserted: List[TMEntry] = []

        async with AsyncSessionLocal() as session:
            try:
                stmt = select(UncertainExample).where(
                    UncertainExample.id == example_id,
                    UncertainExample.status == "pending",
                )
                example = (await session.execute(stmt)).scalar_one_or_none()
                if example is None:
                    logger.warning(
                        "Pending example %s not found for confirmation", example_id
                    )
                    raise UncertainExampleNotFoundError(
                        "Pending example not found",
                        context={"example_id": example_id},
                    )

                entry = TMEntry(
                    en=example.source_text,
                    tgt=corrected_translation,
                    source_lang=example.source_lang,
                    target_lang=example.target_lang,
                )

                inserted = await tm_manager.insert_in_session([entry], session)

                example.corrected_translation = corrected_translation
                example.status = "confirmed"
                example.confirmed_at = datetime.now(timezone.utc)

                await session.commit()

            except UncertainExampleNotFoundError:
                raise
            except Exception as e:
                logger.error(
                    "Failed to confirm correction: %s", e, exc_info=True
                )
                await session.rollback()
                raise

        await tm_manager.index_after_insert(inserted)

        if inserted:
            from backend.worker.tasks import index_translation_memory
            index_translation_memory.delay()

        logger.info(
            "Correction confirmed for example %s (tm_inserted=%d)",
            example_id,
            len(inserted),
        )
        return len(inserted) > 0

    @staticmethod
    async def reject_example(example_id: str) -> bool:
        """
        Mark a pending example as rejected.

        Raises:
            UncertainExampleNotFoundError: no pending example with that id.
        """
        async with AsyncSessionLocal() as session:
            try:
                stmt = select(UncertainExample).where(
                    UncertainExample.id == example_id,
                    UncertainExample.status == "pending",
                )
                example = (await session.execute(stmt)).scalar_one_or_none()
                if example is None:
                    logger.warning(
                        "Pending example %s not found for rejection", example_id
                    )
                    raise UncertainExampleNotFoundError(
                        "Pending example not found",
                        context={"example_id": example_id},
                    )

                example.status = "rejected"
                await session.commit()
                logger.info("Example %s rejected", example_id)
                return True

            except UncertainExampleNotFoundError:
                raise
            except Exception as e:
                logger.error("Failed to reject example: %s", e, exc_info=True)
                await session.rollback()
                raise

    @staticmethod
    async def get_stats() -> Dict[str, int]:
        async with AsyncSessionLocal() as session:
            try:
                stmt = select(
                    func.sum(
                        case((UncertainExample.status == "pending", 1), else_=0)
                    ).label("pending"),
                    func.sum(
                        case((UncertainExample.status == "confirmed", 1), else_=0)
                    ).label("confirmed"),
                    func.sum(
                        case((UncertainExample.status == "rejected", 1), else_=0)
                    ).label("rejected"),
                )
                row = (await session.execute(stmt)).one()
                return {
                    "pending": row.pending or 0,
                    "confirmed": row.confirmed or 0,
                    "rejected": row.rejected or 0,
                }
            except Exception as e:
                logger.error("Failed to get stats: %s", e, exc_info=True)
                return {"pending": 0, "confirmed": 0, "rejected": 0}
```

### Разбор ключевых решений

**Почему `_to_dict` — module-level функция, а не метод.** Вызывается только в `get_pending_examples`, но могла бы использоваться в других местах. Module-level — проще, чем `@staticmethod` внутри класса (у которого тоже есть смысл, но чуть больше церемоний).

**Почему `_to_dict` возвращает `created_at.isoformat()`, а не `datetime`.** Pydantic умеет сериализовать `datetime`, но `FastAPI` в JSON-ответе тоже. Тем не менее `isoformat()` — явный строковый формат, не зависит от версии Pydantic/FastAPI. Плюс `get_pending_examples` возвращает `List[Dict]`, не Pydantic-модели — для однородности.

**Почему `_to_dict` **не** включает `corrected_translation`, `confirmed_at`.** Пример в статусе `pending` — этих полей ещё нет. Если добавить — будут `None`, шум. Для `confirmed` примеры не отдаются через `get_pending_examples` (только `pending`). `_to_dict` — для UI-очереди, где `pending` важнее.

**Почему `add_uncertain_example` проверяет на дубль по `(source_text, source_lang, target_lang, status="pending")`, а не по `id`.** `id` — `uuid4`, всегда разный. Дубль — это «тот же источник в той же паре всё ещё ждёт разбора». `status="pending"` в условии — важно: если пример уже `confirmed` или `rejected`, новый такой же может снова попасть в очередь (он снова «неуверенный»).

**Почему `.limit(1)` в `select`.** На случай, если в БД по какой-то причине два `pending` с одинаковым `source_text` (race condition до создания UNIQUE, или ручной SQL). `.limit(1)` + `scalar_one_or_none` — детерминированно возвращает один. Без `.limit` был бы `MultipleResultsFound`.

**Почему `return existing.id`, а не `raise DuplicateEntryError`.** `add_uncertain_example` вызывается из `perform_translation` в автоматическом режиме. Если пример уже ждёт — не ошибка, просто «уже в очереди». Возврат `existing.id` — идемпотентность: caller получает id, может использовать для логов, но никаких сайд-эффектов.

**Почему `session.add(example)` + `await session.commit()`, а не flush.** `commit` завершает транзакцию, записывает в БД. `flush` — отправляет INSERT, но транзакция открыта, при выходе из `async with` произойдёт rollback (нет явного `commit`). Для `add` — `commit` обязателен.

**Почему `await session.rollback()` в `except`, а не `pass`.** `session.add(example)` уже мог отправить в БД (pending INSERT). Если потом ошибка — нужно откатить, иначе `async with` при выходе... хотя `async with AsyncSessionLocal() as session` **не** коммитит автоматически (в отличие от Django), он делает close без commit → rollback. Но явный `rollback` — чище и быстрее, чем ждать close.

**Почему `get_pending_examples` — `ORDER BY created_at DESC`.** Свежие сверху — редактор видит то, что только что появилось. `LIMIT/OFFSET` — пагинация. `created_at DESC` — стандарт для «очередь задач».

**Почему `if source_lang:` / `if target_lang:`, а не `if source_lang is not None`.** Пустая строка `""` не имеет смысла как фильтр языка — `if source_lang` (truthy) корректно трактует `""` как «не фильтровать». `None` — тоже «не фильтровать». Так что `if` — правильно.

**Почему `get_pending_examples` при исключении возвращает `[]`, а не raise.** Очередь — read-only, UI её рендерит. Если БД упала — лучше показать «нет примеров», чем 500. В `count_pending` — аналогично (`return 0`).

**Почему `confirm_correction` **не** принимает `session` снаружи.** В отличие от `insert_in_session` (который принимает для атомарности с другим write), `confirm_correction` сам открывает сессию — он **сам** этот другой write. Внешний caller (`hitl.py`) передаёт только параметры.

**Почему `confirm_correction` — `@staticmethod`, а не instance-method.** `ActiveLearningManager` — stateless. Все методы — `@staticmethod`. Единообразие.

**Почему **сначала** `tm_manager.insert_in_session`, **потом** `example.status = "confirmed"`.** Порядок операций в одной транзакции не важен для атомарности, но логически — «сначала вставь запись, потом обнови статус». Если `insert_in_session` упадёт — `example` в памяти не мутирован, при `rollback` ничего не записано.

**Почему `insert_in_session` **не** коммитит (комментарий в коде).** Это **ключевое** решение. `insert_in_session` (Этап 7.3) — «INSERT + возврат списка реально вставленных, **без** commit». Caller (`confirm_correction`) делает `commit` после того, как **оба** write (INSERT в TM + UPDATE status) готовы. Один `commit` покрывает всё. Если бы `insert_in_session` коммитил сам — при падении UPDATE `example.status = "confirmed"` TM-запись осталась бы, но пример — `pending`. Рассинхрон.

**Почему `datetime.now(timezone.utc)`, а не `datetime.utcnow()`.** `utcnow()` deprecated в Python 3.12 (Этап 4.1, та же причина). `now(timezone.utc)` — текущий API. `UncertainExample.confirmed_at` — `DateTime(timezone=True)` (Этап 3), принимает tz-aware.

**Почему `except UncertainExampleNotFoundError: raise`, потом `except Exception: rollback; raise`.** `UncertainExampleNotFoundError` (Этап 2) — бизнес-ошибка, caller (`hitl.py`) превращает её в 404. `rollback` для неё не нужен (транзакция только началась, ничего не записано). Но `raise` — нужно, иначе `except Exception` ниже перехватит. Порядок `except` — от специфичного к общему.

**Почему `except Exception` — `rollback`, потом `raise`.** `INSERT` мог частично выполниться, `UPDATE` — упасть. `rollback` откатывает всё. `raise` — caller получит 500, залогирует с `exc_info=True`.

**Почему `index_after_insert` **после** `async with` (после commit).** `index_after_insert` (Этап 7.3) обновляет in-memory кэш (`self._tm_cache.extend(inserted)`) и добавляет пару в `_bm25_dirty`. Делать это **до** commit — значит, что если commit упадёт, in-memory кэш рассинхронизирован с БД. **После** commit — безопасно: данные уже в БД, кэш обновляется.

**Почему `if inserted: index_translation_memory.delay()`.** Celery-задача `index_translation_memory` (Этап 11) переиндексирует Qdrant для всех TM. Если `inserted` пусто (дубликат) — переиндексация не нужна. `delay()` — асинхронный вызов, не блокирует. `if` — экономия.

**Почему импорт `from backend.worker.tasks import index_translation_memory` **внутри** функции, а не на module-level.** `backend.worker.tasks` импортирует `celery_app` (Этап 11), который создаёт Celery-приложение. Если импортировать на module-level — при импорте `active_learning` создастся Celery-приложение, даже если вы не используете worker (например, в тестах). Inline-импорт — ленивая загрузка. Плюс циклическая зависимость: `worker.tasks` импортирует `services.pipeline` (для `batch_translate`), который импортирует `active_learning` — cycle. Inline — разрывает цикл.

**Почему `return len(inserted) > 0`.** `inserted` — список реально вставленных `TMEntry`. Если `0` — дубликат (TM-запись уже была), пример всё равно `confirmed` (в `update`), но caller (`hitl.py`) знает, что TM **не** пополнилась. `tm_inserted` в ответе — различие для UI.

**Почему `reject_example` **не** удаляет пример, а ставит `status="rejected"`.** Аудит: видно, что пример был, кто-то посмотрел, решил отклонить. `DELETE` потерял бы информацию. `status="rejected"` — правильно.

**Почему `get_stats` — один запрос через `CASE WHEN`, а не три `COUNT`.** Один round-trip к БД vs три. На большом объёме (`uncertain_examples` может быть 10000+) — заметно. `case((cond, 1), else_=0)` — SQL `CASE WHEN status='pending' THEN 1 ELSE 0 END`, `sum` — сумма. `func.sum(...).label("pending")` — именованный результат для `row.pending`.

**Почему `row.pending or 0`.** Если таблица пуста — `sum` вернёт `NULL`. `or 0` — `NULL` → `0`. Без — `pending: None`, тип неправильный.

**Почему `get_stats` при исключении возвращает `{"pending": 0, ...}`, а не raise.** Read-only для UI, падение лучше показать нулями, чем 500.

---

## 10.2. `backend/hitl/feedback.py` — `FeedbackHandler`

**Что это:** прямой путь сохранения корректировки в TM, **без** статуса. Вызывается из `POST /hitl/save-correction` — когда редактор вручную добавляет пару в TM, минуя очередь `uncertain_examples`.

**Зачем это нужно:** два сценария HITL:
1. **Через очередь:** `Critic` поставил низкий балл → `add_uncertain_example` → редактор исправляет → `confirm_correction` (Этап 10.1).
2. **Прямой:** редактор знает, что пара плохая (или хорошая) → `save_correction` → сразу в TM. Без очереди.

`FeedbackHandler` — второй сценарий.

Создайте `backend/hitl/feedback.py`:

```python
# backend/hitl/feedback.py
import logging
from typing import Any, Dict

from sqlalchemy import select

from backend.core.models import TMEntry
from backend.data.translation_memory import tm_manager
from backend.db.models import TMDB
from backend.db.session import AsyncSessionLocal

logger = logging.getLogger(__name__)


class FeedbackHandler:
    @staticmethod
    async def save_correction(
        source_text: str,
        source_lang: str,
        target_lang: str,
        corrected_translation: str,
    ) -> Dict[str, Any]:
        """
        Persist a human correction into translation memory.

        Deduplication happens in two places:

        - here, via a read-only lookup, so the caller can distinguish
          ``"skipped"`` from ``"ok"`` in the response;
        - inside ``tm_manager.add_many``, which performs its own check and
          silently skips entries that already exist.

        Side effects:
            Triggers ``index_translation_memory`` (Celery) only when a
            new TM entry was actually inserted.
        """
        async with AsyncSessionLocal() as session:
            stmt = select(TMDB).where(
                TMDB.source_lang == source_lang,
                TMDB.target_lang == target_lang,
                TMDB.source_text == source_text,
                TMDB.target_text == corrected_translation,
            )
            stmt = stmt.limit(1)
            existing = (await session.execute(stmt)).scalar_one_or_none()
            if existing is not None:
                logger.info("Correction already exists in TM")
                return {"status": "skipped", "message": "Entry already exists"}

        entry = TMEntry(
            en=source_text,
            tgt=corrected_translation,
            source_lang=source_lang,
            target_lang=target_lang,
        )

        try:
            inserted = await tm_manager.add_many([entry])
        except Exception as e:
            logger.error("Failed to save correction: %s", e, exc_info=True)
            raise

        if inserted > 0:
            from backend.worker.tasks import index_translation_memory
            index_translation_memory.delay()
            logger.info(
                "Correction saved to TM, reindex scheduled "
                "(%s->%s, %s... -> %s...)",
                source_lang,
                target_lang,
                source_text[:40],
                corrected_translation[:40],
            )
        else:
            logger.info(
                "Correction insert skipped as duplicate (race with "
                "concurrent request)"
            )

        return {"status": "ok", "message": "Correction saved"}
```

### Разбор ключевых решений

**Почему **два** pre-check — здесь и в `add_many`.** `add_many` (Этап 7.3) уже использует `INSERT ... ON CONFLICT DO NOTHING` — дубликаты тихо пропускаются, `inserted` = 0. Но caller (`hitl.py`) не сможет различить:
- «уже было, мы пропустили» → `{"status": "skipped"}`.
- «только что вставили» → `{"status": "ok"}`.

Read-only pre-check здесь даёт `"skipped"` **до** вызова `add_many`, для UX. Если гонка (concurrent) — `add_many` вернёт 0, `else`-ветка лога, ответ `"ok"` (всё равно ok — «попытка не привела к дубликату»). Не идеально, но приемлемо.

**Почему `.limit(1)` в pre-check.** `TMDB` **не** имеет `UniqueConstraint` (Этап 3) — по контенту. Теоретически могут быть две строки с одинаковыми `source_text/target_text` (если был race). `.limit(1)` + `scalar_one_or_none` — детерминированно. Без `.limit` — `MultipleResultsFound`.

**Почему pre-check **не** открывает транзакцию явно.** `async with AsyncSessionLocal() as session` — сессия на чтение. `SELECT` → close (rollback). Никакого commit не нужно. Минимальное время блокировки.

**Почему `entry = TMEntry(...)` **после** закрытия сессии pre-check.** `entry` — Pydantic, не ORM. Не привязан к сессии. Создаём после `async with` — сессия освобождена.

**Почему `await tm_manager.add_many([entry])`, а не `add_entry`.** `add_many` — единый путь для батча. Для одного `entry` — тоже работает. Плюс `add_many` возвращает `len(inserted)` — можно отличить дубликат (0) от нового (1). `add_entry` возвращает `None` (в референсе `async def add_entry(self, entry) -> None: await self.add_many([entry])`), не различить.

**Почему `except Exception: logger.error(...); raise`.** `add_many` может бросить (БД, Qdrant). Логируем, пробрасываем — `hitl.py` превратит в 500.

**Почему `if inserted > 0: index_translation_memory.delay()`.** Переиндексация Qdrant нужна только для новой записи. Дубликат (inserted=0) — Qdrant уже знает. `delay()` — fire-and-forget, `if` — экономия.

**Почему **другой** подход к атомарности, чем в `confirm_correction`.** `save_correction` **не** пишет в `uncertain_examples` — нет второго write для атомарности. `add_many` со своим `commit` внутри — достаточно. В `confirm_correction` — два write, поэтому `insert_in_session` (без commit) + один общий `commit`.

**Почему возвращает `{"status": "ok"|"skipped", "message": "..."}`.** Два статуса — для UI (Streamlit, Этап 12). `skipped` — «уже было», `ok` — «добавили». `hitl.py` отдаёт как есть.

**Почему лог с `%s` для `source_text[:40]`.** Ограничение — не логировать целый текст (может быть 10 КБ). Первые 40 символов — достаточно для идентификации. `%s`-форматирование — стандарт для logging (строка не форматируется, если уровень ниже).

---

## 10.3. `backend/api/routes/hitl.py` — 5 эндпоинтов

**Что это:** REST-слой над `ActiveLearningManager` и `FeedbackHandler`. Все роуты — `require_editor` (admin или editor).

Создайте `backend/api/routes/hitl.py`:

```python
# backend/api/routes/hitl.py
import logging
from typing import Optional

from fastapi import APIRouter, Depends, HTTPException, Query, status
from pydantic import BaseModel, field_validator

from backend.api.deps import require_editor
from backend.core.exceptions import RUNGException
from backend.core.models import validate_language_code
from backend.hitl.active_learning import ActiveLearningManager
from backend.hitl.feedback import FeedbackHandler

logger = logging.getLogger(__name__)
router = APIRouter(prefix="/hitl", tags=["human-in-the-loop"])


class CorrectionRequest(BaseModel):
    source_text: str
    source_lang: str = "en"
    target_lang: str = "ru"
    corrected_translation: str

    @field_validator("source_text", "corrected_translation")
    @classmethod
    def not_empty(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("Field cannot be empty")
        return v.strip()

    @field_validator("source_lang", "target_lang")
    @classmethod
    def validate_lang(cls, v: str) -> str:
        return validate_language_code(v.lower())


class ConfirmCorrectionRequest(BaseModel):
    corrected_translation: str

    @field_validator("corrected_translation")
    @classmethod
    def not_empty(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("Translation cannot be empty")
        return v.strip()


@router.post("/save-correction", status_code=status.HTTP_201_CREATED)
async def save_correction(
    request: CorrectionRequest,
    _=Depends(require_editor),
):
    try:
        return await FeedbackHandler.save_correction(
            source_text=request.source_text,
            source_lang=request.source_lang,
            target_lang=request.target_lang,
            corrected_translation=request.corrected_translation,
        )
    except RUNGException:
        raise
    except HTTPException:
        raise
    except Exception as e:
        logger.error("Failed to save correction: %s", e, exc_info=True)
        raise HTTPException(status_code=500, detail="Failed to save correction")


@router.get("/uncertain")
async def get_uncertain_examples(
    source_lang: Optional[str] = None,
    target_lang: Optional[str] = None,
    limit: int = Query(10, ge=1, le=100),
    offset: int = Query(0, ge=0),
    _=Depends(require_editor),
):
    try:
        examples = await ActiveLearningManager.get_pending_examples(
            source_lang=source_lang,
            target_lang=target_lang,
            limit=limit,
            offset=offset,
        )
        total = await ActiveLearningManager.count_pending(source_lang, target_lang)
        stats = await ActiveLearningManager.get_stats()
        return {
            "examples": examples,
            "stats": stats,
            "total": total,
            "limit": limit,
            "offset": offset,
        }
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to get uncertain examples: %s", e, exc_info=True)
        raise HTTPException(status_code=500, detail="Failed to get uncertain examples")


@router.post("/uncertain/{example_id}/confirm", status_code=status.HTTP_200_OK)
async def confirm_uncertain_example(
    example_id: str,
    request: ConfirmCorrectionRequest,
    _=Depends(require_editor),
):
    try:
        inserted = await ActiveLearningManager.confirm_correction(
            example_id,
            request.corrected_translation,
        )
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to confirm correction: %s", e, exc_info=True)
        raise HTTPException(status_code=500, detail="Failed to confirm correction")

    if inserted:
        message = "Correction confirmed and added to TM"
    else:
        message = "Correction confirmed; entry already existed in TM"

    return {
        "status": "ok",
        "tm_inserted": inserted,
        "message": message,
    }


@router.post("/uncertain/{example_id}/reject", status_code=status.HTTP_200_OK)
async def reject_uncertain_example(
    example_id: str,
    _=Depends(require_editor),
):
    try:
        await ActiveLearningManager.reject_example(example_id)
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to reject example: %s", e, exc_info=True)
        raise HTTPException(status_code=500, detail="Failed to reject example")

    return {"status": "ok", "message": "Example rejected"}


@router.get("/uncertain/stats")
async def get_uncertain_stats(
    _=Depends(require_editor),
):
    try:
        return await ActiveLearningManager.get_stats()
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to get stats: %s", e, exc_info=True)
        raise HTTPException(status_code=500, detail="Failed to get stats")
```

### Разбор ключевых решений

**Почему `CorrectionRequest` и `ConfirmCorrectionRequest` — две модели, а не одна.** `CorrectionRequest` — для `/save-correction`, где нужны `source_text`, `source_lang`, `target_lang`, `corrected_translation`. `ConfirmCorrectionRequest` — для `/uncertain/{id}/confirm`, где нужен только `corrected_translation` (остальное уже в примере). Разные контракты — разные модели. Одна модель с `Optional` полями была бы менее безопасной (Pydantic не смог бы валидировать «обязательность в контексте»).

**Почему `not_empty` применяется и к `source_text`, и к `corrected_translation`.** Пустые строки бесполезны: `source_text=""` не найдёт в TM, `corrected_translation=""` — пустой перевод. `field_validator` с двумя именами — одна функция для обоих полей.

**Почему `v.strip()` в `not_empty`.** Убирает пробелы по краям. « `"  Hello  "` » становится `"Hello"`. Нормализация — на уровне API, не в менеджере.

**Почему `validate_language_code(v.lower())`.** `validate_language_code` (Этап 2) ищет в `LANGUAGES` (ключи — `en`, `ru`, `udm`, все строчные). Если клиент передал `"EN"` — `.lower()` → `"en"` → валидно. `"Udm"` → `"udm"` → валидно (поддерживает 3-символьные). `"us"` → `"us"` → `ValueError` (не в `LANGUAGES`) → 422.

**Почему `ConfirmCorrectionRequest.not_empty` применяется только к `corrected_translation`.** Это единственное поле. `source_text` не передаётся (берётся из `example` в БД).

**Почему `POST /save-correction` — `201 Created`, а `POST /uncertain/{id}/confirm` — `200 OK`.** `/save-correction` создаёт новую TM-запись → 201. `/confirm` обновляет существующий `uncertain_example` (status pending → confirmed) — это update, не create → 200.

**Почему `/uncertain/{id}/confirm` возвращает 200 в обоих случаях (inserted/dedup).** `inserted=True` — TM-запись создана, пример confirmed. `inserted=False` — TM-запись уже была, пример confirmed. Оба — успех. 200 для обоих. Различие — в `tm_inserted` (bool) и `message`.

**Почему `except RUNGException: raise`, потом `except HTTPException: raise`, потом `except Exception: 500`.** Та же логика, что в `admin.py` (Этап 9.5). `UncertainExampleNotFoundError` (наследник `RUNGException`) → глобальный хендлер → 404 с `code: "uncertain_example_not_found"`. `HTTPException` — если бы `ActiveLearningManager` его бросил (сейчас не бросает, но защита). Остальное — 500 с логом.

**Почему `except RUNGException: raise` **до** `except Exception`.** Порядок `except` в Python — первый подходящий. `UncertainExampleNotFoundError` — наследник `RUNGException` **и** `Exception`. Если бы `except Exception` стоял первым — перехватил бы `UncertainExampleNotFoundError` и превратил в 500. Неправильно. `RUNGException` первым — 404 корректно.

**Почему `_ = Depends(require_editor)` — подчёркивание.** FastAPI передаёт пользователя в параметр, но нам он не нужен (используется только для проверки роли). `_` — стандартное имя для «неиспользуемый».

**Почему `Query(10, ge=1, le=100)` в `/uncertain`.** `limit` — 10 по умолчанию (редактор смотрит пачками), максимум 100 (больше — медленно для UI). `offset` — `ge=0` (отрицательный сдвиг невалиден). `Query` валидирует до вызова функции.

**Почему `/uncertain` возвращает **три** вещи: `examples`, `total`, `stats`.** UI (Streamlit) показывает:
1. Список `examples` (что разбирать).
2. `total` — сколько ещё (пагинация).
3. `stats` — общая картина (`pending`, `confirmed`, `rejected`).

Один запрос — три вещи. Альтернатива (три отдельных запроса) — 3× round-trip. Inefficient.

**Почему `get_uncertain_stats` — отдельный роут, хотя `stats` уже в `/uncertain`.** Streamlit-страница System Health (Этап 12) показывает `stats` без загрузки примеров. Отдельный роут — лёгкий (только `get_stats`).

**Почему `confirm_uncertain_example` — `status.HTTP_200_OK` явно, хотя дефолт тоже 200.** Документирует контракт. Читатель кода сразу видит: «200, не 201». В `reject` — то же.

**Почему `reject_uncertain_example` — `status.HTTP_200_OK`, не 204 No Content.** `reject` возвращает `{"status": "ok", "message": "Example rejected"}` — есть тело. 200 с телом. 204 — без тела, клиент должен сам знать, что успех. `200` — лучше для UI.

---

## 10.4. `backend/api/main.py` — регистрация `hitl.router`

**Что это:** добавить `app.include_router(hitl.router)` в `main.py`.

Откройте `backend/api/main.py`. Найдите блок `app.include_router(...)` (после Этапа 9 там `health`, `auth`, `translate`, `admin`), и обновите:

**1. Импорт — добавить `hitl`:**

```python
from backend.api.routes import admin, auth, health, hitl, translate
```

**2. После `app.include_router(admin.router)` — добавить:**

```python
app.include_router(hitl.router)
```

Порядок (не важен для FastAPI, но для читаемости):

```python
app.include_router(health.router)
app.include_router(auth.router)
app.include_router(translate.router)
app.include_router(admin.router)
app.include_router(hitl.router)
```

### Проверка

```powershell
python -m poetry run python -c "from backend.api.main import app; print([r.path for r in app.routes if '/hitl' in getattr(r, 'path', '')])"
```

**Что вы должны увидеть:**

```
['/hitl/save-correction', '/hitl/uncertain', '/hitl/uncertain/{example_id}/confirm', '/hitl/uncertain/{example_id}/reject', '/hitl/uncertain/stats']
```

---

## 10.5. Проверка связки API ↔ HITL

Убедитесь, что связка работает. **Сначала** — что в `.env` `ACTIVE_LEARNING_ENABLED=true`. Если было `false`, поменяйте и `docker compose restart app`.

Зарегистрируйте редактора и админа (если ещё нет):

```powershell
python -m poetry run python scripts/import/create_admin.py --email admin@rung.local --username admin --password Admin1234
```

Создайте регулярного юзера через API (у него будет роль `user` — не подойдёт для `/hitl/*`). Для `/hitl/*` нужен `admin` или `editor`. Роль `editor` API не создаёт — только через `psql`:

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "UPDATE users SET role='editor' WHERE email='editor@rung.local';"
```

Или вручную вставим editor:

```powershell
python -m poetry run python -c "
import asyncio, uuid
from sqlalchemy import select
from backend.db.models import User
from backend.db.session import AsyncSessionLocal
from backend.utils.security import get_password_hash

async def m():
    async with AsyncSessionLocal() as s:
        existing = (await s.execute(select(User).where(User.email=='editor@rung.local'))).scalar_one_or_none()
        if existing is None:
            s.add(User(id=str(uuid.uuid4()), email='editor@rung.local', username='editor', hashed_password=get_password_hash('Editor1234'), role='editor', is_active=True))
            await s.commit()
            print('editor created')
        else:
            print('editor already exists')
asyncio.run(m())
"
```

### Эксперимент: низкий score → запись в очередь

Возьмите админ-токен:

```powershell
$ADMIN_TOKEN = (curl -s -X POST http://localhost:8000/auth/login -H "Content-Type: application/json" -d "{\"email\":\"admin@rung.local\",\"password\":\"Admin1234\"}" | ConvertFrom-Json).access_token
```

Сделайте `/translate` в режиме с Critic, чтобы `critic_score` был низким (шумный текст или необычная пара — qwen часто ставит 5–6):

```powershell
curl -X POST http://localhost:8000/translate -H "Authorization: Bearer $ADMIN_TOKEN" -H "Content-Type: application/json" -d "{\"text\": \"Hello my dear friend, how are you doing today?\", \"source_lang\": \"en\", \"target_lang\": \"tt\"}"
```

**Что вы должны увидеть** в ответе (`critic_score`, `reasoning_log`):

```json
{"translation":"Сәлам, кадерле дустым, бүген ничек хәлләр?","critic_score":6.5,...,"reasoning_log":[{"agent":"Validator",...},{"agent":"Critic",...},...]}
```

`6.5 < 7` → `add_uncertain_example` сработал.

Проверьте очередь:

```powershell
curl http://localhost:8000/hitl/uncertain -H "Authorization: Bearer $ADMIN_TOKEN"
```

**Что вы должны увидеть:**

```json
{
  "examples": [
    {
      "id": "a1b2c3d4-...",
      "source_text": "Hello my dear friend...",
      "source_lang": "en",
      "target_lang": "tt",
      "machine_translation": "Сәлам, кадерле дустым...",
      "critic_score": 6.5,
      "validation_passed": true,
      "created_at": "2026-09-11T15:30:00"
    }
  ],
  "stats": {"pending": 1, "confirmed": 0, "rejected": 0},
  "total": 1,
  "limit": 10,
  "offset": 0
}
```

### Эксперимент: подтверждение (атомарно)

Скопируйте `id` из ответа. Отправьте исправление:

```powershell
$EXAMPLE_ID = "a1b2c3d4-..."
curl -X POST "http://localhost:8000/hitl/uncertain/$EXAMPLE_ID/confirm" -H "Authorization: Bearer $ADMIN_TOKEN" -H "Content-Type: application/json" -d "{\"corrected_translation\": \"Сәлам, кадерле дустым, бүген хәлләрең ничек?\"}"
```

**Что вы должны увидеть:**

```json
{"status": "ok", "tm_inserted": true, "message": "Correction confirmed and added to TM"}
```

Проверьте TM:

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT COUNT(*) FROM translation_memories WHERE source_text='Hello my dear friend, how are you doing today?';"
```

**Что вы должны увидеть:** `1` — запись добавилась.

Проверьте пример:

```powershell
docker compose exec db psql -U rung_user -d rung_db -c "SELECT status, corrected_translation FROM uncertain_examples WHERE id='$EXAMPLE_ID';"
```

**Что вы должны увидеть:** `confirmed | Сәлам, кадерле дустым, бүген хәлләрең ничек?`.

### Эксперимент: повторное подтверждение

```powershell
curl -X POST "http://localhost:8000/hitl/uncertain/$EXAMPLE_ID/confirm" -H "Authorization: Bearer $ADMIN_TOKEN" -H "Content-Type: application/json" -d "{\"corrected_translation\": \"...\"}"
```

**Что вы должны увидеть:**

```json
{"detail":"Pending example not found","code":"uncertain_example_not_found","context":{"example_id":"a1b2c3d4-..."}}
```

Status **404** — потому что `confirm_correction` ищет `status="pending"`, а пример уже `confirmed`. Это правильно: нельзя переподтвердить уже подтверждённый.

---

## 10.6. Тесты

Тесты скинет пользователь. Здесь — что должно проверяться.

### `tests/unit/test_active_learning.py`

- `add_uncertain_example` создаёт запись, возвращает `id`.
- `add_uncertain_example` повторно с теми же `(source, lang, tgt)` → возвращает **тот же** `id`, не создаёт вторую строку.
- `add_uncertain_example` после `confirm_correction` (status="confirmed") с теми же `(source, lang, tgt)` → **создаёт новую** запись (pending пуст → создаётся).
- `get_pending_examples()` — только `pending`, отсортированы `created_at DESC`.
- `get_pending_examples(source_lang="en")` — фильтр.
- `get_pending_examples(limit=1)` — 1.
- `count_pending()` — точное число.
- `confirm_correction(id, "corrected")`:
  - статус `pending` → `confirmed`;
  - `corrected_translation` записан;
  - `confirmed_at` заполнен;
  - `tm_inserted=True`;
  - в `translation_memories` появилась пара.
- `confirm_correction` при уже существующей TM-паре:
  - `tm_inserted=False`;
  - статус `confirmed`;
  - в TM не появилось новой строки.
- `confirm_correction` для несуществующего `id` → `UncertainExampleNotFoundError` (→ 404).
- `confirm_correction` для уже `confirmed` → `UncertainExampleNotFoundError` (поиск по `status="pending"`).
- `reject_example(id)` — статус `rejected`, TM не пополнена.
- `reject_example` несуществующего → `UncertainExampleNotFoundError`.
- `get_stats()` — `{pending, confirmed, rejected}`; на пустой БД — `{0, 0, 0}`.
- **Атомарность:** monkeypatch `example.status = "confirmed"` на падение (`session.commit` бросает) → `tm_manager.insert_in_session` вызван, но после `rollback` TM-запись **отсутствует** в БД. Проверяется через `SELECT` в тестовой сессии.

**Mock/fixture.** SQLite in-memory (Этап 3), фикстура `session` создаёт таблицы. Для `index_translation_memory.delay()` — `monkeypatch.setattr("backend.worker.tasks.index_translation_memory.delay", lambda: None)` (иначе Celery попытается отправить в redis).

**Тест атомарности — детально.**

```python
async def test_confirm_atomicity(session):
    # Создать pending example
    example_id = await ActiveLearningManager.add_uncertain_example(
        "hello", "en", "ru", "привет", critic_score=5.0,
    )

    # Смокать commit на падение
    original_commit = AsyncSession.commit
    async def failing_commit(self):
        raise RuntimeError("simulated failure")
    monkeypatch.setattr(AsyncSession, "commit", failing_commit)

    with pytest.raises(RuntimeError):
        await ActiveLearningManager.confirm_correction(example_id, "привет!")

    monkeypatch.setattr(AsyncSession, "commit", original_commit)

    # Проверить: TM-запись НЕ создана
    tm_count = await session.scalar(
        select(func.count()).select_from(TMDB).where(TMDB.source_text == "hello")
    )
    assert tm_count == 0

    # Проверить: статус остался pending
    ex = (await session.execute(
        select(UncertainExample).where(UncertainExample.id == example_id)
    )).scalar_one()
    assert ex.status == "pending"
```

**Тонкость:** `confirm_correction` открывает **свою** сессию (`async with AsyncSessionLocal()`), а не берёт тестовую. В тесте `AsyncSessionLocal` должен быть подменён на `async_sessionmaker(test_engine)`. Иначе `confirm_correction` пойдёт в прод-БД. Фикстура `monkeypatch.setattr("backend.hitl.active_learning.AsyncSessionLocal", ...)`.

**Что покрыть из `feedback.py`:**

- `save_correction` для новой пары → `{"status": "ok", "message": "Correction saved"}`, TM пополнена.
- `save_correction` для существующей пары → `{"status": "skipped", "message": "Entry already exists"}`, TM **не** пополнена.
- `save_correction` для существующей пары + гонка (mock `add_many` возвращает 0) → `{"status": "ok"}` без переиндексации (или с — зависит от `inserted`; в коде `if inserted > 0` — без).

### `tests/integration/test_hitl_api.py`

Через `TestClient`:

- `GET /hitl/uncertain` без токена → 401.
- `GET /hitl/uncertain` с токеном `user` → 403.
- `GET /hitl/uncertain` с токеном `editor` → 200, поля `examples/stats/total/limit/offset`.
- `POST /hitl/save-correction` с `editor` → 201.
- `POST /hitl/save-correction` повторно с тем же — `{"status": "skipped"}`.
- `POST /hitl/save-correction` с пустым `source_text` → 422.
- `POST /hitl/save-correction` с невалидным `target_lang="xx"` → 422.
- `POST /hitl/uncertain/{id}/confirm` для pending → 200, `tm_inserted=True`.
- `POST /hitl/uncertain/{id}/confirm` для уже confirmed → 404 `{"detail": "Pending example not found", "code": "uncertain_example_not_found"}`.
- `POST /hitl/uncertain/{id}/confirm` с пустым `corrected_translation` → 422.
- `POST /hitl/uncertain/{id}/reject` → 200.
- `GET /hitl/uncertain/stats` → 200, `{pending, confirmed, rejected}`.
- `POST /translate` с `critic_score < 7` (mock Critic) → `GET /hitl/uncertain` показывает новую запись.

**Фикстура `autouse` для `unload_hf_llm`** (Этап 8.9) — уже должна быть в `conftest.py`.

---

## 10.7. Финальная проверка

```powershell
python -m poetry run pytest tests/ -v
```

Все тесты Этапов 1–10 зелёные.

Финальный чек-лист:

- [ ] `curl http://localhost:8000/health` → 200.
- [ ] `ACTIVE_LEARNING_ENABLED=true` в `.env`.
- [ ] `/translate` в `full` режиме с `critic_score < 7` → `uncertain_examples` пополнилась.
- [ ] `GET /hitl/uncertain` (с editor-токеном) → список.
- [ ] `POST /hitl/uncertain/{id}/confirm` → 200, `tm_inserted: true`, запись в TM.
- [ ] Повторный `confirm` → 404.
- [ ] `POST /hitl/uncertain/{id}/reject` → 200, `status="rejected"`.
- [ ] `POST /hitl/save-correction` дважды → 201, потом `{"status": "skipped"}`.
- [ ] `pytest` — зелёный.

Коммит и merge:

```powershell
git add -A
git commit -m "feat(week-10): complete HITL (queue, confirm/reject, feedback, atomic TM insert)"
git push -u origin week-10
```

---

## Что дальше

В **Этапе 11** — Celery worker и beat. `redis` уже поднят (Этап 8). Задачи:
- `index_translation_memory` — вызывается из `confirm_correction` и `save_correction` (в коде уже есть `from backend.worker.tasks import index_translation_memory; index_translation_memory.delay()` — **это** импорт появится в Этапе 11; до этого в тестах нужно `monkeypatch.setattr`).
- `batch_translate`, `run_evaluation_async`, `clear_old_cache`, `add_glossary_bulk`, `add_tm_bulk`.

Worker — в отдельном контейнере `worker`, beat — в `beat`. `persistent event loop` на процесс (Этап 11.3) — ключевая деталь, без неё `async_engine` падает с `Future attached to a different loop`.

**Важно:** после Этапа 10 импорт `from backend.worker.tasks import index_translation_memory` в `active_learning.py` и `feedback.py` будет падать, пока вы не создадите `backend/worker/tasks.py` (Этап 11). В тестах Этапа 10 — `monkeypatch.setattr` на `index_translation_memory.delay`. В production — модуль появится в Этапе 11, до тех пор `confirm_correction` упадёт с `ImportError` при `inserted > 0`. Если хотите проверить Этап 10 end-to-end **до** Этапа 11 — временно закомментируйте `index_translation_memory.delay()` в `active_learning.py` и `feedback.py`, верните в Этапе 11. **Или** создайте `backend/worker/tasks.py` с фиктивным `index_translation_memory = type('X', (), {'delay': staticmethod(lambda: None)})()` — но это хак, лучше подождать Этап 11.